In [61]:
from typing import Annotated
from dotenv import load_dotenv
load_dotenv()

from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [62]:
from langchain_ollama import ChatOllama

In [63]:
llm = ChatOllama(model="llama3.1:8b")

class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State) -> State:
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [64]:
message = {"role": "user", "content": "Who is the Prime Minister of India? Print only the name"}
# message = {"role": "user", "content": "What is the latest price of MSFT stock?"}
response = graph.invoke({"messages":[message]})

response["messages"]

[HumanMessage(content='Who is the Prime Minister of India? Print only the name', additional_kwargs={}, response_metadata={}, id='9807c656-9ad0-4440-b0c6-34123574a0c9'),
 AIMessage(content='Narendra Modi', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-24T17:37:50.248853075Z', 'done': True, 'done_reason': 'stop', 'total_duration': 177589180, 'load_duration': 119989519, 'prompt_eval_count': 22, 'prompt_eval_duration': 10830000, 'eval_count': 4, 'eval_duration': 21607000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019efab5-a9f6-70b1-98f6-9c2762472ca2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 4, 'total_tokens': 26})]

In [65]:
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit","exit"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)

You:  hi


Bot: How's it going? Is there something I can help you with or would you like to chat?


You:  who made google?


Bot: Google was founded by two Ph.D. students at Stanford University in California, USA:

1. **Larry Page** (born March 26, 1973) - He is the CEO of Alphabet Inc., Google's parent company.
2. **Sergey Brin** (born August 21, 1973) - He is a vice president of technology at Alphabet Inc.

The two met while attending Stanford University in 1995 and were initially working on a research project called "Backrub," which aimed to create a search engine that ranked websites based on their importance. They later dropped the Backrub name and decided to call it Google, which was inspired by the word "googol," a mathematical term for a huge number (1 followed by 100 zeros).

They launched Google from a friend's garage in Menlo Park, California in September 1998. The rest is history!

Would you like to know more about Larry Page and Sergey Brin?


You:  exit
